# Professional Ocular Disease Classification Pipeline
This notebook classifies fundus images into DR, Glaucoma, Normal, and Non-Fundus categories using EfficientNet-B0. It includes automated best-model tracking and detailed confusion matrix reporting.

## 1. Setup and Imports

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from timm import create_model
from sklearn.metrics import confusion_matrix
import os
import shutil
import random
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Training and Validation Logic
Includes logic to save the best model based on validation accuracy and print the confusion matrix for every epoch.

In [4]:
def sync_checkpoint_to_drive(model, optimizer, epoch, val_acc, drive_folder):
    """Saves latest checkpoint to Drive and removes the previous one to save space."""
    os.makedirs(drive_folder, exist_ok=True)
    current_drive_path = os.path.join(drive_folder, f"latest_epoch_{epoch+1}.pth")
    prev_drive_path = os.path.join(drive_folder, f"latest_epoch_{epoch}.pth")

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc': val_acc
    }

    torch.save(checkpoint, current_drive_path)

    if os.path.exists(prev_drive_path):
        os.remove(prev_drive_path)

    print(f"--- Drive Sync: {os.path.basename(current_drive_path)} saved, previous removed ---")

def validate_model(model, val_loader, criterion):
    model.eval()
    running_loss, correct = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = correct.double() / len(val_loader.dataset)
    val_loss = running_loss / len(val_loader.dataset)

    # Confusion Matrix display
    cm = confusion_matrix(all_labels, all_preds)
    print(f"Confusion Matrix:\n{cm}")

    return val_acc, val_loss

## 3. Comparative Evaluation
After training, this cell evaluates both the 'best' and 'last' models to compare results.

In [5]:
def evaluate_checkpoint(checkpoint_path, val_loader, num_classes):
    print(f"\n--- Evaluating Checkpoint: {checkpoint_path} ---")
    eval_model = create_model('efficientnet_b0', pretrained=False, num_classes=num_classes).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    # Handle both full state dicts and partial checkpoint dicts
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

    eval_model.load_state_dict(state_dict)
    acc, _ = validate_model(eval_model, val_loader, nn.CrossEntropyLoss())
    print(f"Final Result Accuracy: {acc:.4f}")

# To run evaluation after training:
# evaluate_checkpoint("checkpoints/best_model.pth", val_loader, 4)
# # Example for Drive path:
# evaluate_checkpoint("/content/drive/My Drive/Ocular_Model_Checkpoints/latest_epoch_15.pth", val_loader, 4)


# Ocular Disease Classification using EfficientNet-B0
This notebook demonstrates a complete pipeline for classifying fundus images into four categories: Diabetic Retinopathy (DR), Glaucoma, Normal, and Non-Fundus images.

## 1. Environment Setup and Data Acquisition
Mounting Google Drive and extracting the compressed datasets.

In [6]:
from google.colab import drive
import os
import shutil
import random

drive.mount('/content/drive')

# Extract primary dataset
!unzip -q /content/drive/My\ Drive/Dataset/updated_augmented_dataset.zip -d /content/dataset

# Extract Non-Fundus dataset
!unzip -q /content/drive/My\ Drive/NonFundus.zip -d /content/dataset/updated_augmented_dataset

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/dataset/updated_augmented_dataset/DR_Presence/DR_Negative/0212dd31f623.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/updated_augmented_dataset/DR_Presence/DR_Negative/02358b47ea89.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/dataset/updated_augmented_dataset/DR_Presence/DR_Negative/02358b47ea89.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: replace /content/dataset/updated_augmented_dataset/content/NonFundus/bicycles_Image_7.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## 2. Data Preprocessing and Filtering
Balancing the 'NonFundus' class and organizing the dataset into a filtered directory structure.

In [7]:
folder = '/content/dataset/updated_augmented_dataset/content/NonFundus'
max_images = 1100

image_files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if len(image_files) > max_images:
    to_remove = len(image_files) - max_images
    files_to_delete = random.sample(image_files, to_remove)
    for f in files_to_delete:
        os.remove(os.path.join(folder, f))
    print(f"✅ Folder now contains exactly {max_images} images.")

# Organize into filtered directory
base_dir = "/content/dataset/updated_augmented_dataset"
filtered_dir = "/content/dataset/filtereddataset"

selected_folders = {
    'Glaucoma': os.path.join(base_dir, 'Glaucoma/Glaucoma_Positive'),
    'DR': os.path.join(base_dir, 'DR_Presence/DR_Positive'),
    'Normal': os.path.join(base_dir, 'Normal'),
    'NonFundus': os.path.join(base_dir,'content/NonFundus')
}

if not os.path.exists(filtered_dir): os.makedirs(filtered_dir)

for class_name, src_path in selected_folders.items():
    dest_path = os.path.join(filtered_dir, class_name)
    if not os.path.exists(dest_path): os.makedirs(dest_path)
    if os.path.exists(src_path):
        for file_name in os.listdir(src_path):
            shutil.copy2(os.path.join(src_path, file_name), os.path.join(dest_path, file_name))

print("Filtered dataset created successfully!")

Filtered dataset created successfully!


## 3. Model Configuration and Data Loading
Initializing PyTorch, defining transforms, and preparing DataLoaders.

In [8]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, random_split
# from torchvision import datasets, transforms
# from timm import create_model
# from sklearn.metrics import confusion_matrix
#
# # GPU Check
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
#
# input_size = 224
# batch_size = 8
#
# data_transforms = {
#     "train": transforms.Compose([
#         transforms.RandomResizedCrop(input_size),
#         transforms.RandomHorizontalFlip(),
#         transforms.ColorJitter(),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
#     ]),
#     "val": transforms.Compose([
#         transforms.Resize(input_size + 20),
#         transforms.CenterCrop(input_size),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
#     ])
# }
#
# dataset = datasets.ImageFolder(filtered_dir)
# train_size = int(0.7 * len(dataset))
# val_size = len(dataset) - train_size
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#
# train_dataset.dataset.transform = data_transforms["train"]
# val_dataset.dataset.transform = data_transforms["val"]
#
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from timm import create_model
from sklearn.metrics import confusion_matrix
import multiprocessing

# GPU Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

input_size = 224
# Scaled up batch size from 8 to 128 to fully saturate the A100 GPU
batch_size = 128

data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize(input_size + 20),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# Load original dataset
dataset = datasets.ImageFolder("/content/dataset/filtereddataset")

# Define split ratios upfront (70% Train, 15% Val, 15% Test)
total_len = len(dataset)
train_size = int(0.70 * total_len)
val_size = int(0.15 * total_len)
test_size = total_len - train_size - val_size

# Split dataset using a fixed seed for reproducible splits
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=generator
)

# Assign correct augmentations/transforms to each split
train_dataset.dataset.transform = data_transforms["train"]
val_dataset.dataset.transform = data_transforms["val"]
test_dataset.dataset.transform = data_transforms["val"]

# Determine optimal CPU workers for parallel data preparation
num_workers = min(4, multiprocessing.cpu_count())

# Define optimized loaders with multiprocessing for high-throughput
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    num_workers=num_workers,
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True,
    num_workers=num_workers,
    persistent_workers=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True,
    num_workers=num_workers,
    persistent_workers=True
)

print(f"Optimized Dataset loaders configured for high-performance hardware:")
print(f"- Batch Size: {batch_size}")
print(f"- CPU Workers: {num_workers}")
print(f"- Train samples: {len(train_dataset)}")
print(f"- Validation samples: {len(val_dataset)}")
print(f"- Test samples: {len(test_dataset)}")

Using device: cuda
Optimized Dataset loaders configured for high-performance hardware:
- Batch Size: 128
- CPU Workers: 4
- Train samples: 6879
- Validation samples: 1474
- Test samples: 1475


## 4. Training and Validation
Defining the training loop and executing the model optimization.

In [10]:
def train_model(model, train_loader, val_loader, num_epochs=15, drive_path='/content/drive/My Drive/Ocular_Checkpoints'):
    best_acc = 0.0
    os.makedirs("model_checkpoints", exist_ok=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct = 0.0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            torch.cuda.empty_cache()

        scheduler.step()

        # Validation & Metrics
        val_acc, _ = validate_model(model, val_loader, criterion)
        train_acc = correct.double() / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        # Local Save (Best Model Only)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "model_checkpoints/best_model.pth")
            print("--- Local Best Model Updated ---")

        # Drive persistence (Helper function)
        sync_checkpoint_to_drive(model, optimizer, epoch, val_acc, drive_path)

# Initialize and run
num_classes = len(dataset.classes)
model = create_model('efficientnet_b0', pretrained=True, num_classes=num_classes).to(device)
train_model(model, train_loader, val_loader)

Confusion Matrix:
[[847   0   0   0]
 [  2 241   0  16]
 [  0   0 187   0]
 [  4   9   0 168]]
Epoch 1/15 | Train Acc: 0.9289 | Val Acc: 0.9790
--- Local Best Model Updated ---
--- Drive Sync: latest_epoch_1.pth saved, previous removed ---
Confusion Matrix:
[[841   1   0   5]
 [  0 255   0   4]
 [  0   0 187   0]
 [  1  22   0 158]]
Epoch 2/15 | Train Acc: 0.9917 | Val Acc: 0.9776
--- Drive Sync: latest_epoch_2.pth saved, previous removed ---
Confusion Matrix:
[[846   0   0   1]
 [  4 241   0  14]
 [  0   0 187   0]
 [  1   6   0 174]]
Epoch 3/15 | Train Acc: 0.9967 | Val Acc: 0.9824
--- Local Best Model Updated ---
--- Drive Sync: latest_epoch_3.pth saved, previous removed ---
Confusion Matrix:
[[847   0   0   0]
 [  0 256   0   3]
 [  0   0 187   0]
 [  6  24   0 151]]
Epoch 4/15 | Train Acc: 0.9856 | Val Acc: 0.9776
--- Drive Sync: latest_epoch_4.pth saved, previous removed ---
Confusion Matrix:
[[846   0   0   1]
 [  0 254   0   5]
 [  0   0 187   0]
 [  2  18   0 161]]
Epoch 5/15 

### 4.1 Evaluation on a Dedicated Test Split
To calculate the final test accuracy, we split our original dataset into Train (70%), Validation (15%), and Test (15%) subsets. Then, we evaluate the best saved checkpoint on the Test split.

## 5. Inference
Loading a trained checkpoint and predicting on a single image.

In [16]:
from PIL import Image

def predict(image_path, model_checkpoint):
    model = create_model('efficientnet_b0', pretrained=False, num_classes=4).to(device)
    model.load_state_dict(torch.load(model_checkpoint, map_location=device))
    model.eval()

    preprocess = transforms.Compose([
        transforms.Resize(244), # Adjusted to training size
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
        pred_idx = torch.argmax(probs).item()

    class_mapping = {0: "DR", 1: "Glaucoma", 2: "NonFundus", 3: "Normal"}
    print(f"Predicted: {class_mapping[pred_idx]} ({probs[pred_idx].item()*100:.2f}%)")


In [17]:
predict("/content/images.jpg", "/content/model_checkpoints/best_model.pth")

Predicted: NonFundus (100.00%)


In [20]:
# Evaluate the best model directly from Google Drive on the Test Split
drive_checkpoint = "/content/model_checkpoints/best_model.pth"

evaluate_checkpoint(drive_checkpoint, test_loader, num_classes=4)


--- Evaluating Checkpoint: /content/model_checkpoints/best_model.pth ---
Confusion Matrix:
[[921   0   0   0]
 [  0 267   0   3]
 [  0   0 146   0]
 [  0  10   0 128]]
Final Result Accuracy: 0.9912
